# 🧠 Tier 2 DRQN Training — Cognitive Radar Interception

**Team Cache No Money — Person 3 (Deep RL Engineer)**

This notebook runs the full Tier 2 training pipeline on Google Colab's free T4 GPU:

1. ✅ Clone repo & install dependencies
2. ✅ Download dataset shard from HuggingFace
3. ✅ Run unit tests to verify pipeline
4. ✅ Stage 8A: Supervised pre-training (next-channel prediction)
5. ✅ Stage 8B: Online Double-DQN training in RadarEnv
6. ✅ Plot training curves & benchmark against Tier 1 bandits
7. ✅ Export checkpoint to Google Drive

---

> **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## Cell 1 — Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    !nvidia-smi
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## Cell 2 — Clone Repository & Install Dependencies

Clones branch `Part-3_DL_Layer` from `eld-dlh/Cache_No_Money`.

In [ ]:
import os

# ──────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/eld-dlh/Cache_No_Money.git"
BRANCH = "Part-3_DL_Layer"
# ──────────────────────────────────────────────────

REPO_NAME = "Cache_No_Money"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH} {GITHUB_REPO}
    print(f"✅ Cloned {REPO_NAME} (branch: {BRANCH})")
else:
    print(f"✅ {REPO_NAME} already exists. Pulling latest...")
    !cd {REPO_NAME} && git checkout {BRANCH} && git pull

%cd {REPO_NAME}

# Install dependencies
!pip install -q -r requirements.txt
print("\n✅ Dependencies installed.")

## Cell 3 — Download Dataset & Generate Binary

Downloads a sample from the Alan Turing Synthetic Radar Dataset (or activates the multi-emitter coherent generator) and converts it to the 32-byte `.npy` binary format.

In [ ]:
from pathlib import Path

NPY_PATH = Path("data/raw/pdw_records.npy")

if NPY_PATH.exists():
    print(f"✅ Binary file already exists: {NPY_PATH}")
    print(f"   Size: {NPY_PATH.stat().st_size / 1e6:.1f} MB")
else:
    print("📥 Running data pipeline...")
    print("\n── Step 1: Download dataset shard")
    !python data/download_dataset.py
    
    print("\n── Step 2: Parse PDW records")
    !python data/parse_pdw.py
    
    print("\n── Step 3: Convert to binary")
    !python data/convert_to_binary.py
    
    if NPY_PATH.exists():
        print(f"\n✅ Binary file created: {NPY_PATH}")
        print(f"   Size: {NPY_PATH.stat().st_size / 1e6:.1f} MB")
    else:
        print("\n❌ Binary file was not created. Check the output above for errors.")

## Cell 4 — Run Unit Tests

Validates all Person 3 components (network, buffer, agent, controller) using synthetic data.

In [ ]:
!python tests/test_drqn_pipeline.py

## Cell 5 — Train the DRQN Agent

Runs the full two-stage training pipeline:
- **Stage 8A:** Supervised pre-training (5 epochs) achieves ~80% initial accuracy
- **Stage 8B:** Online Double-DQN fine-tuning with protected best-checkpoint

Adjust `--rl-steps` to control training duration:
- `5000` for a quick test (~3 min on T4)
- `20000` for a solid run (~12 min on T4)
- `30000` for deep fine-tuning (~18 min on T4)

In [ ]:
# ──────────────────────────────────────────────────
# ⚙️ TRAINING CONFIGURATION — Edit as needed
RL_STEPS = 20000          # Total environment steps
PRETRAIN_EPOCHS = 5       # Supervised pre-training epochs
BATCH_SIZE = 32           # Replay buffer batch size
EVAL_EVERY = 2000         # Evaluate on test split every N steps
# ──────────────────────────────────────────────────

!python train_drqn.py \
    --device cuda \
    --rl-steps {RL_STEPS} \
    --pretrain-epochs {PRETRAIN_EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --eval-every {EVAL_EVERY} \
    --log-every 200

## Cell 6 — Plot Training Curves & Benchmark vs Tier 1 Bandits

Visualises key interception metrics comparing DRQN against Tier 1 bandit baselines (Random, ε-Greedy, UCB1, SW-UCB).

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, ".")

from env.radar_env import RadarEnv, N_CHANNELS
from models.drqn_agent import DRQNAgent
from models.state_builder import StateBuilder

# Load best checkpoint
ckpt_path = Path("checkpoints/drqn_radar_best.pt")
npy_path = Path("data/raw/pdw_records.npy")

if ckpt_path.exists() and npy_path.exists():
    agent = DRQNAgent(n_actions=N_CHANNELS, device="cpu")
    agent.load(ckpt_path, load_optimiser=False)
    agent.set_eval_mode()

    state_builder = StateBuilder(n_channels=N_CHANNELS, max_steps=500)
    episode_rewards = []
    episode_hit_rates = []

    print("\n── Running 5 evaluation episodes on test split...")
    for ep in range(5):
        env = RadarEnv(npy_path, split="test", max_steps=500, seed=100 + ep)
        obs, info = env.reset()
        state_builder.reset()

        init_info = {"intercepted": False, "chosen_channel": 0,
                     "pulse_channel": 0, "step": 0}
        state = state_builder.build_state(obs, init_info)
        ep_reward, ep_hits, ep_steps = 0.0, 0, 0

        for step in range(500):
            action, _ = agent.select_action(state, None, evaluate=True)
            obs, reward, terminated, truncated, info = env.step(action)
            state = state_builder.build_state(obs, info)
            ep_reward += reward
            ep_hits += int(info.get("intercepted", False))
            ep_steps += 1
            if terminated or truncated:
                break

        episode_rewards.append(ep_reward)
        episode_hit_rates.append(ep_hits / max(ep_steps, 1))
        env.close()

    drqn_hit_rate = float(np.mean(episode_hit_rates) * 100)
    drqn_color = '#2ECC71' if drqn_hit_rate >= 55.0 else '#E05252'

    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart: DRQN vs Bandit baselines
    labels = ['Random\n(1.56%)', 'ε-Greedy\n(28.6%)', 'UCB1\n(44.9%)',
              'SW-UCB\n(56.0%)', 'DRQN\n(Ours)']
    hit_rates = [1.56, 28.56, 44.90, 55.98, drqn_hit_rate]
    colors = ['#666666', '#4A90D9', '#5BA55B', '#E8A838', drqn_color]

    axes[0].bar(labels, hit_rates, color=colors, edgecolor='white', linewidth=1.5)
    axes[0].set_ylabel('Interception Rate (%)', fontsize=12)
    axes[0].set_title('Hit Rate: DRQN vs Tier 1 Bandits', fontsize=14)
    axes[0].set_ylim(0, 100)
    for i, v in enumerate(hit_rates):
        axes[0].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold')

    # Per-episode rewards
    axes[1].bar(range(1, len(episode_rewards) + 1), episode_rewards,
               color=drqn_color, edgecolor='white')
    axes[1].set_xlabel('Episode', fontsize=12)
    axes[1].set_ylabel('Total Episode Reward', fontsize=12)
    axes[1].set_title('DRQN Test Episode Rewards', fontsize=14)

    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 DRQN Test Results:")
    print(f"   Avg hit rate:       {drqn_hit_rate:.2f}%")
    print(f"   Avg episode reward: {np.mean(episode_rewards):.2f}")
else:
    print("⚠️  Missing data or checkpoint. Run Cells 3 and 5 first.")

## Cell 7 — Save Checkpoint to Google Drive

Saves the trained model weights to your Google Drive for safekeeping and for Person 4 to use.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
src_ckpt = Path("checkpoints/drqn_radar_best.pt")
dst_dir = Path("/content/drive/MyDrive/Cache_No_Money_Checkpoints")
dst_dir.mkdir(parents=True, exist_ok=True)
dst_ckpt = dst_dir / "drqn_radar_best.pt"

if src_ckpt.exists():
    shutil.copy2(src_ckpt, dst_ckpt)
    print(f"✅ Checkpoint saved to Google Drive:")
    print(f"   {dst_ckpt}")
    print(f"   Size: {dst_ckpt.stat().st_size / 1e6:.1f} MB")
    
    # Also save the results plot if it exists
    results_img = Path("training_results.png")
    if results_img.exists():
        shutil.copy2(results_img, dst_dir / "training_results.png")
        print(f"   Also saved: training_results.png")
else:
    print("❌ No checkpoint found. Run Cell 5 first.")

---

## ✅ Done!

Your trained DRQN checkpoint is now saved at:
- **In this session:** `checkpoints/drqn_radar_best.pt`
- **In Google Drive:** `My Drive/Cache_No_Money_Checkpoints/drqn_radar_best.pt`

### Next Steps
- Push the checkpoint back to the GitHub repo for Person 4
- Compare the DRQN hit rate against Person 2's SW-UCB baseline (~56%)
- Hand off `checkpoints/drqn_radar_best.pt` and `models/` to Person 4 (Systems Engineer)